# Tissue novelty detection (public data)

Reduce features to 24 principal components using known classes only. Fit one shrinkage covariance model to all known examples, then one per known class. Compare pooled Mahalanobis distance with distance to the closest class. A larger score indicates a more unusual sample. Use up to 650 training tiles per known class and the official PathMNIST test set. Class 8 (colorectal adenocarcinoma epithelium) is excluded from training and is the positive class at test. Features summarize spatial color and multiscale texture. This is novelty detection for one withheld tissue type.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'tissue-novelty-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import hashlib\nimport os\nfrom pathlib import Path\n\nDATA = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\nfrom urllib.request import urlretrieve\n\np = DATA / "medmnist/pathmnist.npz"\np.parent.mkdir(parents=True, exist_ok=True)\nif not p.exists():\n    urlretrieve("https://zenodo.org/records/10519652/files/pathmnist.npz?download=1", p)\nassert hashlib.md5(p.read_bytes()).hexdigest() == "a8b06965200029087d5bd730944a56c1"\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef describe(images):\n    from scipy.ndimage import gaussian_filter\n\n    a = images.astype("float32") / 255.0\n    blocks = a.reshape(len(a), 4, 7, 4, 7, 3).mean((2, 4)).reshape(len(a), -1)\n    texture = []\n    for s in [1.0, 2.0]:\n        residual = np.abs(a - gaussian_filter(a, (0, s, s, 0)))\n        texture.append(residual.mean((1, 2)))\n        texture.append(residual.std((1, 2)))\n    return np.column_stack([blocks, *texture])\n\n\ndef dataset(seed):\n    with np.load(DATA / "medmnist/pathmnist.npz") as raw:\n        y = raw["train_labels"].ravel()\n        t = raw["test_labels"].ravel()\n        rng = np.random.default_rng(seed)\n        ids = np.concatenate(\n            [\n                rng.choice(\n                    np.flatnonzero(y == k), min(700, (y == k).sum()), replace=False\n                )\n                for k in range(8)\n            ]\n        )\n        x = describe(raw["train_images"][ids])\n        z = describe(raw["test_images"])\n    return x, y[ids], z, t\n\n\nfrom sklearn.covariance import LedoitWolf\nfrom sklearn.decomposition import PCA\n\n\ndef run(seed):\n    x, y, z, t = dataset(seed)\n    rng = np.random.default_rng(seed)\n    ids = np.concatenate(\n        [\n            rng.choice(np.flatnonzero(y == k), min(650, (y == k).sum()), replace=False)\n            for k in np.unique(y)\n            if k != 8\n        ]\n    )\n    encoder = make_pipeline(StandardScaler(), PCA(n_components=24, random_state=seed))\n    a = encoder.fit_transform(x[ids])\n    b = encoder.transform(z)\n    pooled = LedoitWolf().fit(a)\n    classes = [LedoitWolf().fit(a[y[ids] == k]) for k in np.unique(y[ids])]\n    save((encoder, pooled, classes), f"covariance-{seed}")\n    return {\n        "pooled": scores(t == 8, pooled.mahalanobis(b)),\n        "class_conditional": scores(\n            t == 8, np.min([m.mahalanobis(b) for m in classes], axis=0)\n        ),\n    }\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())